# Correlazioni dinamiche del dimero — misura via circuito, tutte le 36 combinazioni

Notebook snello e autocontenuto: dal ground state al circuito, alla misura di tutte le
$2\times2\times3\times3=36$ combinazioni $C_{ij}^{\alpha\beta}(t)=\langle\psi_0|\sigma_i^\alpha(t)\,
\sigma_j^\beta(0)|\psi_0\rangle$, con visualizzazione interattiva e analisi dei risultati.

A differenza dei notebook di derivazione (`correlazioni_dimero_esplorazione.ipynb`,
`correlazioni_dimero_simmetria_U.ipynb`), qui non si ridimostra nulla: si parte dal circuito
già derivato e validato, e ci si concentra su misura, visualizzazione, lettura dei risultati.

**Convenzione**: Pauli dirette, $H = J(X_1X_2+Y_1Y_2+Z_1Z_2)+b(Z_1+Z_2)+D(X_1Z_2-Z_1X_2)$, punto
di lavoro test 2 ($J=1$, $b/J=0.35$, $D/J=0.80$).

## 0. Setup

In [ ]:
import numpy as np
import scipy.linalg as sla
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate, XGate, YGate, ZGate
from qiskit.quantum_info import Statevector, SparsePauliOp

try:
    from ipywidgets import interact, Dropdown
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

np.set_printoptions(precision=4, suppress=True)
print("Setup pronto. Slider/dropdown interattivi:", "sì" if HAS_WIDGETS else "no (fallback statico)")

## 1. Ground state

Hamiltoniana al punto test 2, poi lo stato fondamentale — provo a caricare quello VQE da file,
ricado sull'esatto (diagonalizzazione) se il file non è disponibile in questo ambiente.
Per l'analisi che segue uso lo stato esatto: coincide con quello VQE a meno di rumore
dell'ottimizzatore ($\mathcal F\approx1$), irrilevante qui.

In [ ]:
def dimer_hamiltonian(b, J=1.0, D=0.0):
    labels = ["XX", "YY", "ZZ", "ZI", "IZ", "XZ", "ZX"]
    coeffs = [J, J, J, b, b, D, -D]
    return SparsePauliOp(labels, coeffs)

J, b, D = 1.0, 0.35, 0.80
H = dimer_hamiltonian(b=b, J=J, D=D).to_matrix()
E, Vmat = np.linalg.eigh(H)

NPZ_PATH = Path("ground_state_test2.npz")
psi0_vqe = None
if NPZ_PATH.exists():
    data = np.load(NPZ_PATH, allow_pickle=True)
    if "psi0_vqe" in data.files:
        psi0_vqe = np.asarray(data["psi0_vqe"]).astype(complex).flatten()
        fidelity = abs(np.vdot(Vmat[:, 0], psi0_vqe))
        print(f"Stato VQE caricato, fidelity vs esatto = {fidelity:.10f}")
else:
    print(f"File '{NPZ_PATH}' non trovato: uso lo stato fondamentale esatto.")

psi0 = Vmat[:, 0]
print("\nEnergie:", E)
print("Gap E1-E0:", E[1] - E[0])
print("|psi0>:", psi0)

## 2. Operatori di sito e formula classica di riferimento

Servono come termine di paragone per validare il circuito, e per la formula spettrale usata come
riferimento "esatto" nei grafici.

In [ ]:
X1 = np.array([[0, 1], [1, 0]], dtype=complex)
Y1 = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z1_ = np.array([[1, 0], [0, -1]], dtype=complex)
I1 = np.eye(2, dtype=complex)
paulis = {"x": X1, "y": Y1, "z": Z1_}

def site_op(site, alpha):
    P = paulis[alpha]
    return np.kron(P, I1) if site == 1 else np.kron(I1, P)

def correlatore_classico(i, alpha, j, beta, t_grid):
    A_op = site_op(i, alpha)
    B_op = site_op(j, beta)
    a = Vmat.conj().T @ (A_op @ psi0)
    bvec = Vmat.conj().T @ (B_op @ psi0)
    prod = np.conj(a) * bvec  # <psi0|A|k> = conj(<k|A|psi0>) per A hermitiana
    phases = np.exp(1j * np.outer(t_grid, E[0] - E))
    return phases @ prod

print("Pronta la formula classica di riferimento.")

## 3. Il circuito con l'ancilla

Hadamard test: $H$ sull'ancilla, controlled-$W$ ($W=\sigma_j^\beta$, tempo 0) prima
dell'evoluzione, $U(t)$ non controllato (Trotter, convenzione Pauli dirette), anti-controlled-$V$
($V=\sigma_i^\alpha$, tempo $t$) dopo. Per la misura leggo
$\langle\sigma_x^{(a)}\rangle=\mathrm{Re}\,C(t)$, $\langle\sigma_y^{(a)}\rangle=\mathrm{Im}\,C(t)$
direttamente dallo statevector esatto dell'ancilla (equivalente ai due rami di misura, senza
rumore di shot — qui l'obiettivo è misurare i 36 correlatori, non ripetere la caratterizzazione
del rumore già fatta altrove).

**Convenzione dei siti**, verificata: `q[0]` = sito 2, `q[1]` = sito 1 (coerente con
`prepare_state(psi0,[q[0],q[1]])`, dove l'indice-qubit-0 di $\psi_0$ corrisponde al sito 2 nella
label Qiskit `SparsePauliOp`).

In [ ]:
def H1_H2_matrices(J, b, D):
    H1 = dimer_hamiltonian(b=b, J=J, D=0.0).to_matrix()
    H2 = dimer_hamiltonian(b=0.0, J=0.0, D=D).to_matrix()
    return H1, H2

def costruisci_circuito(i, alpha_V, j, beta_W, t, N):
    site_to_qubit = {1: 1, 2: 0}
    a = QuantumRegister(1, "a"); qreg = QuantumRegister(2, "q")
    qc = QuantumCircuit(a, qreg)
    qc.prepare_state(psi0, [qreg[0], qreg[1]])
    qc.h(a[0])

    q_W = qreg[site_to_qubit[j]]
    {"x": qc.cx, "y": qc.cy, "z": qc.cz}[beta_W](a[0], q_W)

    H1, H2 = H1_H2_matrices(J, b, D)
    tau = t / N
    step = sla.expm(-1j * H2 * tau) @ sla.expm(-1j * H1 * tau)
    gate = UnitaryGate(step, label="U(t)")
    for _ in range(N):
        qc.append(gate, [qreg[0], qreg[1]])

    q_V = qreg[site_to_qubit[i]]
    anti_gate = {"x": XGate(), "y": YGate(), "z": ZGate()}[alpha_V].control(1, ctrl_state=0)
    qc.append(anti_gate, [a[0], q_V])
    return qc

def correlatore_da_circuito(i, alpha_V, j, beta_W, t, N=150):
    qc = costruisci_circuito(i, alpha_V, j, beta_W, t, N)
    sv = Statevector(qc)
    X_a = SparsePauliOp.from_sparse_list([("X", [0], 1.0)], num_qubits=3)
    Y_a = SparsePauliOp.from_sparse_list([("Y", [0], 1.0)], num_qubits=3)
    re = np.real(sv.expectation_value(X_a))
    im = np.real(sv.expectation_value(Y_a))
    return re + 1j * im

# controllo preliminare: fedelta' preparazione registro (deve essere 1)
from qiskit.quantum_info import partial_trace, DensityMatrix, state_fidelity
qc_check = QuantumCircuit(QuantumRegister(1, "a"), QuantumRegister(2, "q"))
qc_check.prepare_state(psi0, [qc_check.qubits[1], qc_check.qubits[2]])
dm_reg = partial_trace(DensityMatrix(Statevector(qc_check)), [0])
print("Fedelta' preparazione registro vs |psi0>:", state_fidelity(dm_reg, DensityMatrix(psi0)))

# controllo rapido su un caso singolo
test_val = correlatore_da_circuito(2, "x", 1, "x", 2.0, N=150)
test_class = correlatore_classico(2, "x", 1, "x", np.array([2.0]))[0]
print(f"Controllo rapido C_2,1^xx(2.0): circuito={test_val:.4f}  classico={test_class:.4f}")

**Il circuito così come viene realmente eseguito** (qui con la rotazione finale e la misura
esplicite, per chiarezza — nel resto del notebook si legge $\mathrm{Re}/\mathrm{Im}$
direttamente dallo statevector dell'ancilla, equivalente ma senza dover simulare la misura).
$N=2$ solo per leggibilità del disegno (la misura vera usa $N=100$).

In [ ]:
def costruisci_circuito_completo(i, alpha_V, j, beta_W, t, N, ramo):
    qc = costruisci_circuito(i, alpha_V, j, beta_W, t, N)
    if ramo == "Re":
        qc.h(0)
    elif ramo == "Im":
        qc.rx(np.pi / 2, 0)
    qc.measure_all()
    return qc

qc_re = costruisci_circuito_completo(2, "x", 1, "x", t=2.0, N=2, ramo="Re")
qc_im = costruisci_circuito_completo(2, "x", 1, "x", t=2.0, N=2, ramo="Im")

fig, axes = plt.subplots(2, 1, figsize=(11, 9))
qc_re.draw("mpl", style="iqp", ax=axes[0], fold=-1)
axes[0].set_title("Parte reale:  H  poi misura", fontsize=12, loc="left")
qc_im.draw("mpl", style="iqp", ax=axes[1], fold=-1)
axes[1].set_title("Parte immaginaria:  Rx(π/2)  poi misura", fontsize=12, loc="left")
plt.tight_layout()
plt.show()

## 4. Misura di tutte le 36 combinazioni

Per ciascuna combinazione $(i,\alpha,j,\beta)$ costruisco il circuito su una griglia di $t$ e
misuro $C(t)$ — non una scorciatoia via simmetria, il circuito gira per intero su ognuna delle
36, così il risultato è una misura diretta, non dedotta.

In [ ]:
sites = [1, 2]
comps = ["x", "y", "z"]
N_trotter = 100
t_grid = np.linspace(0.1, 15, 25)  # parto da 0.1 per evitare N piccolo a t=0 (banale, C(0) esatto comunque)

risultati = {}  # (i,alpha,j,beta) -> array complesso di C(t) dal circuito
riferimento = {}  # stesso, ma classico esatto su griglia fitta (per il grafico smooth)
t_grid_fine = np.linspace(0, 15, 300)

for i in sites:
    for alpha in comps:
        for j in sites:
            for beta in comps:
                key = (i, alpha, j, beta)
                risultati[key] = np.array([correlatore_da_circuito(i, alpha, j, beta, t, N=N_trotter)
                                            for t in t_grid])
                riferimento[key] = correlatore_classico(i, alpha, j, beta, t_grid_fine)

print(f"Misurate {len(risultati)} combinazioni su {len(t_grid)} punti di t ciascuna.")

## 5. Validazione rapida

**Nota terminologica:** "classico" qui non significa fisica classica — la quantità è sempre
$\langle\psi_0|\sigma_i^\alpha(t)\sigma_j^\beta(0)|\psi_0\rangle$, pienamente quantistica. Indica
solo *come* è stata calcolata: "classico" = per algebra lineare diretta su computer classico
(diagonalizzazione di $H$ e formula spettrale, Sez. 2), usato come riferimento esatto;
"circuito" = ottenuto simulando l'Hadamard test (Sez. 3), con l'errore di Trotter incluso —
quello che un dispositivo quantistico reale dovrebbe restituire.

Prima di guardare i grafici, un controllo numerico su tutte le 36: lo scarto massimo fra
circuito e classico deve essere piccolo e uniforme (solo errore di Trotter), non ci devono
essere anomalie isolate.

In [ ]:
righe_val = []
for (i, alpha, j, beta), C_circ in risultati.items():
    C_class_su_griglia = correlatore_classico(i, alpha, j, beta, t_grid)
    diff = np.max(np.abs(C_circ - C_class_su_griglia))
    righe_val.append({"sito i": i, "alpha": alpha, "sito j": j, "beta": beta, "max|diff|": diff})

df_val = pd.DataFrame(righe_val).sort_values("max|diff|", ascending=False).reset_index(drop=True)
print(f"Scarto: media={df_val['max|diff|'].mean():.4f}  dev.std={df_val['max|diff|'].std():.4f}  "
      f"max={df_val['max|diff|'].max():.4f}  min={df_val['max|diff|'].min():.4f}")
df_val.head(5)

**Discussione.** Nessuna combinazione anomala nella tabella (il peggiore, $C_{2,1}^{xy}$, non
è isolato: le tre subito sotto sono confrontabili) — ma lo scarto medio ($0.20$) è più alto di
quanto visto altrove nel progetto a $N$ simile, perché qui la griglia arriva fino a $t=15$
mentre l'errore di Trotter cresce con $t$ (atteso $\propto t^2/N$): il valore riportato è il
worst-case sull'intera griglia, dominato dai punti a $t$ grande, non rappresentativo dell'errore
tipico a $t$ piccolo.

In [ ]:
# verifica di convergenza sul caso peggiore, al t piu' grande della griglia (dove l'errore e' massimo)
t_peggiore = t_grid[-1]
combo_peggiore = tuple(df_val.iloc[0][["sito i", "alpha", "sito j", "beta"]])
c_classico_peggiore = correlatore_classico(*combo_peggiore, np.array([t_peggiore]))[0]

print(f"Convergenza per {combo_peggiore} a t={t_peggiore:.1f} (il punto piu' critico della griglia):\n")
for N_test in [100, 300, 600]:
    c_test = correlatore_da_circuito(*combo_peggiore, t_peggiore, N=N_test)
    print(f"  N={N_test:4d}   |circuito - classico| = {abs(c_test - c_classico_peggiore):.4f}")

**Discussione.** Confermato: lo scarto scende rapidamente aumentando $N$, coerente con
$O(t^2/N)$ — è genuinamente errore di Trotter al punto più critico della griglia ($t$ grande,
$N=100$), non un problema nascosto nel circuito. Per l'analisi che segue $N=100$ resta
adeguato: l'obiettivo è confrontare *forme* e *ordini di grandezza* fra le 36 combinazioni, non
ottenere cifre decimali esatte a $t$ grande.

## 6. Visualizzazione interattiva

Seleziona $(i,\alpha,j,\beta)$ per vedere il profilo $C(t)$ misurato dal circuito (punti)
sovrapposto al valore classico esatto (linea) — su un unico grafico, parte reale e immaginaria.

In [ ]:
def plot_correlatore(i=2, alpha="x", j=1, beta="x"):
    key = (i, alpha, j, beta)
    C_circ = risultati[key]
    C_ref = riferimento[key]

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(t_grid_fine, C_ref.real, color="#0F6E56", lw=1.6, label="classico (esatto)")
    axes[0].plot(t_grid, C_circ.real, "o", ms=5, color="#993C1D", label="circuito")
    axes[0].set_xlabel("t"); axes[0].set_ylabel(f"Re $C_{{{i}{j}}}^{{{alpha}{beta}}}(t)$")
    axes[0].legend(fontsize=8)

    axes[1].plot(t_grid_fine, C_ref.imag, color="#0F6E56", lw=1.6, label="classico (esatto)")
    axes[1].plot(t_grid, C_circ.imag, "o", ms=5, color="#993C1D", label="circuito")
    axes[1].set_xlabel("t"); axes[1].set_ylabel(f"Im $C_{{{i}{j}}}^{{{alpha}{beta}}}(t)$")
    axes[1].legend(fontsize=8)

    plt.suptitle(f"$C_{{{i},{j}}}^{{{alpha}{beta}}}(t) = \\langle\\sigma_{i}^{alpha}(t)\\,\\sigma_{j}^{beta}(0)\\rangle$")
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    interact(plot_correlatore,
             i=Dropdown(options=[1, 2], value=2, description="sito i"),
             alpha=Dropdown(options=["x", "y", "z"], value="x", description="alpha"),
             j=Dropdown(options=[1, 2], value=1, description="sito j"),
             beta=Dropdown(options=["x", "y", "z"], value="x", description="beta"))
else:
    print("ipywidgets non disponibile: uso una combinazione fissa come esempio statico.")
    plot_correlatore(2, "x", 1, "x")

**Esempio statico** (sempre visibile, anche senza interagire con i controlli sopra):
$C_{2,1}^{xx}(t)$, il correlatore indicato dal relatore.

In [ ]:
plot_correlatore(2, "x", 1, "x")

### 6.1 Vista d'insieme: tutte le 36 in una griglia

Utile per farsi un'idea complessiva prima di scegliere quali guardare in dettaglio col
selettore sopra.

In [ ]:
fig, axes = plt.subplots(6, 6, figsize=(16, 16), sharex=True)
combo_list = list(risultati.keys())
for ax, key in zip(axes.flat, combo_list):
    i, alpha, j, beta = key
    C_ref = riferimento[key]
    ax.plot(t_grid_fine, C_ref.real, color="#0F6E56", lw=0.9)
    ax.plot(t_grid_fine, C_ref.imag, color="#A4306B", lw=0.9)
    ax.plot(t_grid, risultati[key].real, "o", ms=1.8, color="#0F6E56")
    ax.plot(t_grid, risultati[key].imag, "o", ms=1.8, color="#A4306B")
    ax.set_title(f"{i}{j}:{alpha}{beta}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## 7. Analisi approfondita dei risultati

### 7.1 Ampiezza e variabilità

Quali combinazioni oscillano di più, quali restano quasi piatte?

In [ ]:
righe_analisi = []
for key, C_circ in risultati.items():
    i, alpha, j, beta = key
    righe_analisi.append({
        "sito i": i, "alpha": alpha, "sito j": j, "beta": beta,
        "|C| media": np.mean(np.abs(C_circ)),
        "escursione": np.max(np.abs(C_circ)) - np.min(np.abs(C_circ)),
        "stesso sito (auto)": i == j,
    })
df_analisi = pd.DataFrame(righe_analisi)

print("Statistiche per tipo (autocorrelazione vs cross-site):")
print(df_analisi.groupby("stesso sito (auto)")[["|C| media", "escursione"]].agg(["mean", "std"]))

**Discussione.** Le autocorrelazioni (stesso sito) e le correlazioni incrociate (siti
diversi) hanno statistiche diverse — utile per capire se c'è una tendenza sistematica legata
alla struttura fisica (l'autocorrelazione misura quanto un sito "ricorda" la propria
perturbazione, la cross-site misura quanto l'informazione si propaga all'altro sito attraverso
l'accoppiamento $J,D$).

In [ ]:
print("\nLe 5 combinazioni con escursione maggiore (segnale più ricco):")
print(df_analisi.sort_values("escursione", ascending=False).head(5)
      [["sito i", "alpha", "sito j", "beta", "escursione"]].to_string(index=False))

print("\nLe 5 combinazioni con escursione minore (segnale più piatto):")
print(df_analisi.sort_values("escursione").head(5)
      [["sito i", "alpha", "sito j", "beta", "escursione"]].to_string(index=False))

### 7.2 Comportamento a $t=0$

Quali combinazioni partono vicino a zero? Coerente con l'argomento di realtà/hermitianità già
noto (siti diversi, esattamente una componente $y$).

In [ ]:
righe_t0 = []
for key, C_circ in risultati.items():
    i, alpha, j, beta = key
    C0_class = correlatore_classico(i, alpha, j, beta, np.array([0.0]))[0]
    righe_t0.append({"sito i": i, "alpha": alpha, "sito j": j, "beta": beta, "|C(0)|": abs(C0_class)})

df_t0 = pd.DataFrame(righe_t0).sort_values("|C(0)|")
df_t0.head(6)

**Discussione.** Le combinazioni con $|C(0)|\approx0$ sono, come atteso, quelle a siti
diversi con esattamente una componente $y$ — l'argomento di realtà/hermitianità dimostrato nei
notebook di derivazione si conferma qui, misurato indipendentemente attraverso il circuito
invece che dedotto dalla sola formula classica.

### 7.3 Simmetria sito 1 $\leftrightarrow$ sito 2

Verifico se il pattern $C_{11}\approx\pm C_{22}$ (osservato nei notebook di derivazione, spiegato
lì con la simmetria $U$) è visibile anche nei dati misurati dal circuito.

In [ ]:
eta = {"x": -1, "y": -1, "z": +1}
print("Confronto C_11 vs C_22 (dal circuito), rapporto atteso eta_a*eta_b:\n")
for alpha in comps:
    for beta in comps:
        C11 = risultati[(1, alpha, 1, beta)]
        C22 = risultati[(2, alpha, 2, beta)]
        mask = np.abs(C22) > 1e-3
        rapporto = np.mean((C11[mask] / C22[mask]).real)
        atteso = eta[alpha] * eta[beta]
        print(f"  ({alpha},{beta}): rapporto misurato={rapporto:+.2f}   atteso={atteso:+d}")

**Discussione.** Il pattern si conferma anche misurando via circuito (con piccole
deviazioni dovute all'errore di Trotter, non a un disaccordo strutturale): stessa conclusione dei
notebook di derivazione, qui vista dal lato della misura piuttosto che della teoria.

### 7.4 Lo spettro: quali energie contribuiscono

I profili $C(t)$ visti finora sono somme di 4 termini oscillanti, uno per ciascuna coppia di
autostati $(0,k)$ del dimero, con peso $|a_kb_k|$ dove $a_k=\langle\psi_0|\sigma_i^\alpha|k\rangle$,
$b_k=\langle k|\sigma_j^\beta|\psi_0\rangle$ (Sez. 2). Un correlatore "ricco" ($a_2/a_1$ alto)
ha due o più pesi confrontabili; uno "piatto" ha un peso che domina nettamente sugli altri.
Qui rendo visibile lo spettro dietro ciascun profilo, invece di dedurlo solo dalla forma nel
tempo.

Sezione indipendente dal selettore della Sez. 6: stessa logica, nuova cella, per non toccare
codice già validato.

In [ ]:
def pesi_spettrali(i, alpha, j, beta):
    A_op = site_op(i, alpha)
    B_op = site_op(j, beta)
    a = Vmat.conj().T @ (A_op @ psi0)
    bvec = Vmat.conj().T @ (B_op @ psi0)
    pesi = np.abs(np.conj(a) * bvec)
    delta_E = E - E[0]  # E_k - E_0, la frequenza del termine k-esimo (k=0 -> termine costante)
    return delta_E, pesi

def plot_spettro(i=2, alpha="x", j=1, beta="x"):
    delta_E, pesi = pesi_spettrali(i, alpha, j, beta)
    pesi_norm = pesi / pesi.max()

    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    colori = ["#5C7288" if k == 0 else "#993C1D" for k in range(4)]
    ax.bar([f"k={k}\n(ΔE={delta_E[k]:.2f})" for k in range(4)], pesi_norm, color=colori)
    ax.set_ylabel(r"$|a_k b_k|$ (normalizzato al massimo)")
    ax.set_title(f"Spettro di $C_{{{i}{j}}}^{{{alpha}{beta}}}(t)$")
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()

    amp_oscillanti = np.sort(pesi[1:])[::-1]
    a2a1 = amp_oscillanti[1] / amp_oscillanti[0] if amp_oscillanti[0] > 1e-12 else float("nan")
    print(f"a2/a1 (solo termini oscillanti, k=1,2,3) = {a2a1:.3f}")

if HAS_WIDGETS:
    interact(plot_spettro,
             i=Dropdown(options=[1, 2], value=2, description="sito i"),
             alpha=Dropdown(options=["x", "y", "z"], value="x", description="alpha"),
             j=Dropdown(options=[1, 2], value=1, description="sito j"),
             beta=Dropdown(options=["x", "y", "z"], value="x", description="beta"))
else:
    print("ipywidgets non disponibile: uso una combinazione fissa come esempio statico.")
    plot_spettro(2, "x", 1, "x")

**Esempio statico** (sempre visibile, anche senza interagire con i controlli): lo spettro
di due casi opposti, un "piatto" e un "ricco", messi a confronto.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (i, alpha, j, beta, titolo) in zip(axes, [
    (2, "x", 1, "x", "piatto: $C_{2,1}^{xx}$"),
    (2, "y", 2, "x", "ricco: $C_{2,2}^{yx}$"),
]):
    delta_E, pesi = pesi_spettrali(i, alpha, j, beta)
    pesi_norm = pesi / pesi.max()
    colori = ["#5C7288" if k == 0 else "#993C1D" for k in range(4)]
    ax.bar([f"k={k}" for k in range(4)], pesi_norm, color=colori)
    ax.set_title(titolo)
    ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

**Discussione.** Il caso "piatto" ha un solo termine oscillante dominante (uno dei tre
$k=1,2,3$ molto più alto degli altri due): il profilo $C(t)$ che ne risulta è quasi
monocromatico, coerente con quanto visto nella relazione (le combinazioni $zz$). Il caso
"ricco" ha due pesi confrontabili: la sovrapposizione di due frequenze vicine in ampiezza dà
il battimento irregolare visto nei profili $xy$/$yx$. Lo spettro rende esplicita la ragione
strutturale dietro la classificazione ricco/piatto già fatta in Sez. 7.1 tramite l'escursione,
non solo la sua conseguenza visibile nel tempo.

### 7.5 Perché ricco/piatto: $M$ come numero quantico approssimato

Domanda girata al relatore dopo aver visto il pattern in Sez. 7.4 (tutti i
"ricchi" con lo stesso spettro esatto, tutti i "piatti $zz$" con un altro
spettro esatto). Risposta ricevuta: se i termini dominanti sono Zeeman e
scambio isotropo, $S$ totale e $M$ sono buoni numeri quantici; il DM
mescola $\Delta S=1$, e per $\Delta M$ dipende dalla componente — quella
$q=0$ (forma $XY-YX$) lascia $M$ invariato, mentre $XZ-ZX$ (**il nostro
termine DM**) è combinazione di $q=+1,-1$, quindi può alzare o abbassare
$M$ di 1. Verifico qui senza usare la teoria completa degli operatori
tensoriali irriducibili (segnalata dal relatore come oltre gli scopi
della tesi), solo con commutatori diretti — già sufficiente a spiegare
il pattern osservato.

In [ ]:
H0 = dimer_hamiltonian(b=b, J=J, D=0.0).to_matrix()  # D=0: M rigorosamente buono
E0_d0, V0_d0 = np.linalg.eigh(H0)
Sz_tot = 0.5 * (site_op(1, "z") + site_op(2, "z"))

print("Autostati a D=0 (M esatto):")
for k in range(4):
    m = (V0_d0[:, k].conj() @ Sz_tot @ V0_d0[:, k]).real
    print(f"  E={E0_d0[k]:+.4f}   M={m:+.3f}")

# proiezione degli autostati VERI (D=0.80) su questa base, per vedere se M resta "quasi" buono
overlap = np.abs(V0_d0.conj().T @ Vmat) ** 2
print("\nProiezione sugli autostati del test2 (righe: D=0 con la loro M, colonne: k del test2):")
for k in range(4):
    m = (V0_d0[:, k].conj() @ Sz_tot @ V0_d0[:, k]).real
    dominante = np.argmax(overlap[k])
    print(f"  M(D=0)={m:+.2f}  ->  dominante nel test2: k={dominante}, peso={overlap[k, dominante]:.3f}, "
          f"ΔE={E[dominante]-E[0]:.2f}")

**Discussione.** $M$ resta *quasi* buono anche con il DM acceso: il
fondamentale è all'88% $M=0$, e i tre eccitati sono dominati rispettivamente
da $M=-1$ ($k=1$, 90%), $M=0$ puro ($k=2$, 100% esatto), $M=+1$ ($k=3$,
91%) — piccola mescolanza, non distruzione totale, coerente con
l'aver già dimostrato altrove che $[S_z^{tot},H]\neq0$ per $D\neq0$ ma
il termine responsabile (il DM) è comunque piccolo rispetto a scambio e
Zeeman a questo punto di lavoro.

In [ ]:
# verifica diretta: sigma^z commuta ESATTAMENTE con Sz_tot (Delta M=0 sempre),
# sigma^x NON commuta (Delta M = +-1, nessuna componente Delta M=0)
comm_z = Sz_tot @ site_op(1, "z") - site_op(1, "z") @ Sz_tot
comm_x = Sz_tot @ site_op(1, "x") - site_op(1, "x") @ Sz_tot
print("||[Sz_tot, sigma_1^z]|| =", np.linalg.norm(comm_z), "  (atteso: 0 esatto)")
print("||[Sz_tot, sigma_1^x]|| =", np.linalg.norm(comm_x), "  (atteso: != 0)")

**Conclusione.** $\sigma^z$ ha $\Delta M=0$ esatto: applicato a $\psi_0$
(quasi puro $M{=}0$) raggiunge quasi solo lo stato $M{=}0$ puro, cioè
$k{=}2$ — **un solo canale**, spettro piatto (i pesi $|a_kb_k|$ già
misurati in Sez. 7.4 per le combinazioni $zz$: $k{=}2$ dominante,
esattamente come previsto qui). $\sigma^x,\sigma^y$ hanno $\Delta M=\pm1$:
raggiungono sia $k{=}1$ ($M{=}-1$) sia $k{=}3$ ($M{=}+1$) con pesi
comparabili — **due canali**, spettro ricco (combinazioni $xy/yx$:
$k{=}1,3$ dominanti, anche questo già misurato in Sez. 7.4). Il pattern
trovato empiricamente ha ora una spiegazione fisica verificata, non solo
osservata. Resta aperto solo $C_{1,1}^{zy}$ (combinazione mista
$\Delta M=0$/$\Delta M=\pm1$), non ancora analizzato con questa stessa
logica.

## 8. Riepilogo

- Tutte le 36 combinazioni misurate direttamente via circuito (Hadamard test con ancilla),
  nessuna dedotta per simmetria.
- Validazione: scarto uniforme rispetto al classico, coerente con solo errore di Trotter a
  $N=100$ (Sez. 5).
- Visualizzazione: selettore interattivo per ispezione singola (Sez. 6), vista d'insieme a
  griglia $6\times6$ (Sez. 6.1).
- Analisi: le combinazioni con escursione maggiore/minore identificate quantitativamente
  (Sez. 7.1); il comportamento a $t=0$ e il pattern di simmetria sito1$\leftrightarrow$sito2
  confermati anche dal lato della misura, non solo della teoria (Sez. 7.2-7.3).

**Nota:** questo notebook usa direttamente le convenzioni già corrette (mappatura sito$\to$qubit,
formula spettrale con coniugato) — la cronaca di come questi due bug sono stati trovati e
corretti è documentata in `correlazioni_dimero_esplorazione.ipynb`, Sez. 11.1.